# How many distinct values does each column take, and what does that imply about its type?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [37]:
import os
from pathlib import Path

import polars as pl
from IPython.display import Markdown, display

pl.Config.set_tbl_rows(100)
pl.Config.set_fmt_str_lengths(50)

def get_raw_dir():
    current = Path(os.getcwd())
    while current.parent != current:
        if (current / "kaggle" / "raw").exists():
            return current / "kaggle" / "raw"
        current = current.parent
    return Path("../../kaggle/raw")

raw_dir = get_raw_dir()
df_txn = pl.scan_csv(raw_dir / "train_transaction.csv", infer_schema_length=10000, null_values=[""])
df_id = pl.scan_csv(raw_dir / "train_identity.csv", infer_schema_length=10000, null_values=[""])

df = df_txn.join(df_id, on="TransactionID", how="left").collect()

### Unique Values and Type Analysis

Infer the logical type of each column based on its content, overriding the default autodetection where possible (e.g. integer vs float, boolean vs integer).

In [34]:
def infer_optimal_type(col_series: pl.Series) -> dict:
    null_count = col_series.null_count()
    total_count = len(col_series)
    is_nullable = null_count > 0
    
    s_valid = col_series.drop_nulls()
    if len(s_valid) == 0:
        return {"inferred_type": "unknown (all nulls)", "nullable": is_nullable, "example_values": None}
        
    dtype = col_series.dtype
    inferred_type = str(dtype)
    example_values = s_valid.unique().head(5).to_list()
    
    if dtype in [pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Int16, pl.Int8, pl.UInt64, pl.UInt32, pl.UInt16, pl.UInt8]:
        try:
            is_integer = (s_valid == s_valid.cast(pl.Int64)).all() if dtype in [pl.Float64, pl.Float32] else True
        except:
            is_integer = False
            
        if is_integer:
            if s_valid.n_unique() <= 2 and s_valid.min() >= 0 and s_valid.max() <= 1:
                inferred_type = "boolean"
            else:
                inferred_type = "integer"
        else:
            inferred_type = "float"
            
    elif dtype == pl.Boolean:
        inferred_type = "boolean"
    elif dtype == pl.Utf8 or dtype == pl.String:
        if s_valid.n_unique() <= 2:
            unique_vals = set(s_valid.str.to_lowercase().unique().to_list())
            if unique_vals.issubset({'t', 'f'}) or unique_vals.issubset({'true', 'false'}) or unique_vals.issubset({'y', 'n'}) or unique_vals.issubset({'yes', 'no'}):
                inferred_type = "boolean (from string)"
            else:
                inferred_type = "string"
        else:
            inferred_type = "string"
            
    return {
        "Column": col_series.name,
        "Inferred Type": inferred_type,
        "Nullable": is_nullable,
        "Null Percent (%)": round(null_count / total_count * 100, 2),
        "Example Values": str(example_values)
    }

results = [infer_optimal_type(df[col]) for col in df.columns]
df_types = pl.DataFrame(results).sort(by=["Inferred Type", "Null Percent (%)", "Column"])

List of columns categorized by inferred type.

In [38]:
def inspect_block(prefix):
    return (
        df_types.filter(pl.col("Column").str.starts_with(prefix))
        # .group_by(["Inferred Type", "Nullable"])
        # .agg(pl.len().alias("Count"))
        # .sort(by=["Inferred Type", "Nullable"])
    )

Breakdown of column blocks by inferred types and nullability.

In [48]:
Markdown(inspect_block("addr")
         .to_pandas()
         .to_markdown(index=False))

| Column   | Inferred Type   | Nullable   |   Null Percent (%) | Example Values                      |
|:---------|:----------------|:-----------|-------------------:|:------------------------------------|
| addr1    | integer         | True       |              11.13 | [100.0, 101.0, 102.0, 104.0, 105.0] |
| addr2    | integer         | True       |              11.13 | [10.0, 13.0, 14.0, 15.0, 16.0]      |

In [47]:
Markdown(inspect_block("card")
         .to_pandas()
         .to_markdown(index=False))

| Column   | Inferred Type   | Nullable   |   Null Percent (%) | Example Values                                         |
|:---------|:----------------|:-----------|-------------------:|:-------------------------------------------------------|
| card1    | integer         | False      |               0    | [1000, 1001, 1004, 1005, 1006]                         |
| card3    | integer         | True       |               0.27 | [100.0, 101.0, 102.0, 105.0, 106.0]                    |
| card5    | integer         | True       |               0.72 | [100.0, 101.0, 102.0, 104.0, 105.0]                    |
| card2    | integer         | True       |               1.51 | [100.0, 101.0, 102.0, 103.0, 104.0]                    |
| card4    | string          | True       |               0.27 | ['visa', 'mastercard', 'american express', 'discover'] |
| card6    | string          | True       |               0.27 | ['charge card', 'debit', 'debit or credit', 'credit']  |

In [49]:
Markdown(inspect_block("C")
         .to_pandas()
         .to_markdown(index=False))

| Column   | Inferred Type   | Nullable   |   Null Percent (%) | Example Values            |
|:---------|:----------------|:-----------|-------------------:|:--------------------------|
| C1       | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C10      | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C11      | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C12      | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C13      | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C14      | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C2       | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C3       | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C4       | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C5       | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C6       | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C7       | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C8       | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| C9       | integer         | False      |                  0 | [0.0, 1.0, 2.0, 3.0, 4.0] |

In [50]:
Markdown(inspect_block("D")
         .to_pandas()
         .to_markdown(index=False))

| Column     | Inferred Type   | Nullable   |   Null Percent (%) | Example Values                                                              |
|:-----------|:----------------|:-----------|-------------------:|:----------------------------------------------------------------------------|
| D8         | float           | True       |              87.31 | [0.0, 0.04166600108146668, 0.08333300054073334, 0.125, 0.16666600108146667] |
| D9         | float           | True       |              87.31 | [0.0, 0.04166600108146668, 0.08333300054073334, 0.125, 0.16666600108146667] |
| D1         | integer         | True       |               0.21 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                   |
| D10        | integer         | True       |              12.87 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                   |
| D15        | integer         | True       |              15.09 | [-83.0, -74.0, -60.0, -53.0, -30.0]                                         |
| D4         | integer         | True       |              28.6  | [-122.0, -90.0, -83.0, -74.0, -53.0]                                        |
| D3         | integer         | True       |              44.51 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                   |
| D11        | integer         | True       |              47.29 | [-53.0, -33.0, -29.0, -28.0, -15.0]                                         |
| D2         | integer         | True       |              47.55 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                   |
| D5         | integer         | True       |              52.47 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                   |
| D6         | integer         | True       |              87.61 | [-83.0, -74.0, -6.0, 0.0, 1.0]                                              |
| D12        | integer         | True       |              89.04 | [-83.0, -74.0, 0.0, 1.0, 2.0]                                               |
| D14        | integer         | True       |              89.47 | [-193.0, -83.0, 0.0, 1.0, 2.0]                                              |
| D13        | integer         | True       |              89.51 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                   |
| D7         | integer         | True       |              93.41 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                   |
| DeviceType | string          | True       |              76.16 | ['mobile', 'desktop']                                                       |
| DeviceInfo | string          | True       |              79.91 | ['BBB100-1', 'GT-I8200N', 'TA-1028 Build/NMF26O', 'Fractal', 'ME173X']      |

In [52]:
Markdown(inspect_block("dist")
         .to_pandas()
         .to_markdown(index=False))

| Column   | Inferred Type   | Nullable   |   Null Percent (%) | Example Values            |
|:---------|:----------------|:-----------|-------------------:|:--------------------------|
| dist1    | integer         | True       |              59.65 | [0.0, 1.0, 2.0, 3.0, 4.0] |
| dist2    | integer         | True       |              93.63 | [0.0, 1.0, 2.0, 3.0, 4.0] |

In [51]:
Markdown(inspect_block("M")
         .to_pandas()
         .to_markdown(index=False))

| Column   | Inferred Type         | Nullable   |   Null Percent (%) | Example Values     |
|:---------|:----------------------|:-----------|-------------------:|:-------------------|
| M6       | boolean (from string) | True       |              28.68 | ['F', 'T']         |
| M1       | boolean (from string) | True       |              45.91 | ['F', 'T']         |
| M2       | boolean (from string) | True       |              45.91 | ['F', 'T']         |
| M3       | boolean (from string) | True       |              45.91 | ['T', 'F']         |
| M8       | boolean (from string) | True       |              58.63 | ['T', 'F']         |
| M9       | boolean (from string) | True       |              58.63 | ['F', 'T']         |
| M7       | boolean (from string) | True       |              58.64 | ['F', 'T']         |
| M5       | boolean (from string) | True       |              59.35 | ['T', 'F']         |
| M4       | string                | True       |              47.66 | ['M1', 'M2', 'M0'] |

In [31]:
v_summary = inspect_block("V")
display(Markdown(v_summary.to_pandas().to_markdown(index=False)))

| Column   | Inferred Type   | Nullable   |   Null Percent (%) | Example Values                                                                         |
|:---------|:----------------|:-----------|-------------------:|:---------------------------------------------------------------------------------------|
| V107     | boolean         | True       |               0.05 | [0.0, 1.0]                                                                             |
| V14      | boolean         | True       |              12.88 | [0.0, 1.0]                                                                             |
| V65      | boolean         | True       |              13.06 | [0.0, 1.0]                                                                             |
| V88      | boolean         | True       |              15.1  | [0.0, 1.0]                                                                             |
| V41      | boolean         | True       |              28.61 | [0.0, 1.0]                                                                             |
| V1       | boolean         | True       |              47.29 | [0.0, 1.0]                                                                             |
| V306     | float           | True       |               0    | [0.0, 0.2915000021457672, 0.4668999910354614, 0.5830000042915344, 0.7547000050544739]  |
| V307     | float           | True       |               0    | [0.0, 0.2915000021457672, 0.4668999910354614, 0.5426999926567078, 0.5830000042915344]  |
| V308     | float           | True       |               0    | [0.0, 0.2915000021457672, 0.4668999910354614, 0.5426999926567078, 0.5830000042915344]  |
| V309     | float           | True       |               0    | [0.0, 0.2915000021457672, 1.0, 1.6404000520706177, 1.7383999824523926]                 |
| V310     | float           | True       |               0    | [0.0, 0.2915000021457672, 0.4839000105857849, 0.5426999926567078, 0.6873999834060669]  |
| V311     | float           | True       |               0    | [0.0, 0.2915000021457672, 1.0, 1.7383999824523926, 1.8602999448776243]                 |
| V312     | float           | True       |               0    | [0.0, 0.2915000021457672, 0.5426999926567078, 0.8607000112533569, 1.0]                 |
| V316     | float           | True       |               0    | [0.0, 0.2915000021457672, 0.4668999910354614, 0.5830000042915344, 0.6413000226020813]  |
| V317     | float           | True       |               0    | [0.0, 0.2915000021457672, 0.4668999910354614, 0.5830000042915344, 0.604200005531311]   |
| V318     | float           | True       |               0    | [0.0, 0.2915000021457672, 0.4668999910354614, 0.5830000042915344, 0.6413000226020813]  |
| V319     | float           | True       |               0    | [0.0, 1.993899941444397, 2.1517999172210693, 2.2578001022338867, 2.2960000038146973]   |
| V320     | float           | True       |               0    | [0.0, 1.3085999488830566, 1.993899941444397, 2.2578001022338867, 2.291699886322021]    |
| V321     | float           | True       |               0    | [0.0, 1.3085999488830566, 1.993899941444397, 2.1517999172210693, 2.2578001022338867]   |
| V126     | float           | True       |               0.05 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.4668999910354614, 0.7547000050544739] |
| V127     | float           | True       |               0.05 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.4668999910354614, 0.7547000050544739] |
| V128     | float           | True       |               0.05 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.4668999910354614, 0.7547000050544739] |
| V129     | float           | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 3.3145999908447266]                                               |
| V130     | float           | True       |               0.05 | [0.0, 1.0, 2.0, 2.537600040435791, 3.0]                                                |
| V131     | float           | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 3.3145999908447266]                                               |
| V132     | float           | True       |               0.05 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.4668999910354614, 0.7547000050544739] |
| V133     | float           | True       |               0.05 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.4668999910354614, 0.7547000050544739] |
| V134     | float           | True       |               0.05 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.4668999910354614, 0.7547000050544739] |
| V135     | float           | True       |               0.05 | [0.0, 2.1517999172210693, 2.2960000038146973, 2.336199998855591, 2.582200050354004]    |
| V136     | float           | True       |               0.05 | [0.0, 2.1517999172210693, 2.2960000038146973, 2.336199998855591, 2.582200050354004]    |
| V137     | float           | True       |               0.05 | [0.0, 2.1517999172210693, 2.2960000038146973, 2.336199998855591, 2.582200050354004]    |
| V313     | float           | True       |               0.21 | [0.0, 0.2915000021457672, 0.4839000105857849, 0.5426999926567078, 0.6873999834060669]  |
| V314     | float           | True       |               0.21 | [0.0, 0.2915000021457672, 0.4839000105857849, 0.5426999926567078, 0.6873999834060669]  |
| V315     | float           | True       |               0.21 | [0.0, 0.2915000021457672, 0.4839000105857849, 0.5426999926567078, 0.6873999834060669]  |
| V270     | float           | True       |              76.05 | [0.0, 0.2915000021457672, 1.170199990272522, 1.7383999824523926, 1.8602999448776243]   |
| V271     | float           | True       |              76.05 | [0.0, 0.2915000021457672, 0.498199999332428, 1.170199990272522, 1.7383999824523926]    |
| V272     | float           | True       |              76.05 | [0.0, 0.2915000021457672, 0.498199999332428, 1.170199990272522, 1.7383999824523926]    |
| V208     | float           | True       |              76.32 | [0.0, 0.2915000021457672, 1.170199990272522, 1.6404000520706177, 1.7383999824523926]   |
| V209     | float           | True       |              76.32 | [0.0, 0.2915000021457672, 1.170199990272522, 1.6404000520706177, 1.7383999824523926]   |
| V210     | float           | True       |              76.32 | [0.0, 0.2915000021457672, 1.170199990272522, 1.6404000520706177, 1.7383999824523926]   |
| V202     | float           | True       |              76.36 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.5830000042915344, 0.8479999899864197] |
| V203     | float           | True       |              76.36 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.5830000042915344, 0.8479999899864197] |
| V204     | float           | True       |              76.36 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.5830000042915344, 0.8479999899864197] |
| V205     | float           | True       |              76.36 | [0.0, 0.2915000021457672, 1.170199990272522, 1.6404000520706177, 1.7383999824523926]   |
| V206     | float           | True       |              76.36 | [0.0, 0.2915000021457672, 1.7383999824523926, 1.8602999448776243, 1.895799994468689]   |
| V207     | float           | True       |              76.36 | [0.0, 0.2915000021457672, 1.170199990272522, 1.6404000520706177, 1.7383999824523926]   |
| V211     | float           | True       |              76.36 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.8479999899864197, 0.8781999945640564] |
| V212     | float           | True       |              76.36 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.8479999899864197, 0.8781999945640564] |
| V213     | float           | True       |              76.36 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.8479999899864197, 0.8781999945640564] |
| V214     | float           | True       |              76.36 | [0.0, 1.3085999488830566, 1.993899941444397, 2.1517999172210693, 2.2578001022338867]   |
| V215     | float           | True       |              76.36 | [0.0, 1.3085999488830566, 1.993899941444397, 2.1517999172210693, 2.175100088119507]    |
| V216     | float           | True       |              76.36 | [0.0, 1.3085999488830566, 1.993899941444397, 2.1517999172210693, 2.2578001022338867]   |
| V263     | float           | True       |              77.91 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.5830000042915344, 0.8781999945640564] |
| V264     | float           | True       |              77.91 | [0.0, 0.2915000021457672, 0.5702999830245972, 0.5830000042915344, 0.604200005531311]   |
| V265     | float           | True       |              77.91 | [0.0, 0.10000000149011612, 0.2899999916553497, 0.2915000021457672, 0.5702999830245972] |
| V266     | float           | True       |              77.91 | [0.0, 0.2915000021457672, 1.170199990272522, 1.7383999824523926, 1.8602999448776243]   |
| V267     | float           | True       |              77.91 | [0.0, 0.2915000021457672, 0.498199999332428, 0.5702999830245972, 0.7300000190734863]   |
| V268     | float           | True       |              77.91 | [0.0, 0.10000000149011612, 0.2899999916553497, 0.2915000021457672, 0.5702999830245972] |
| V269     | float           | True       |              77.91 | [0.0, 5.0, 6.0, 7.0, 7.5]                                                              |
| V273     | float           | True       |              77.91 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.6413000226020813, 0.8781999945640564] |
| V274     | float           | True       |              77.91 | [0.0, 0.2915000021457672, 0.604200005531311, 0.6413000226020813, 0.8781999945640564]   |
| V275     | float           | True       |              77.91 | [0.0, 0.2915000021457672, 0.42399999499320973, 0.6413000226020813, 0.8781999945640564] |
| V276     | float           | True       |              77.91 | [0.0, 1.3085999488830566, 1.993899941444397, 2.1517999172210693, 2.2578001022338867]   |
| V277     | float           | True       |              77.91 | [0.0, 1.3085999488830566, 1.5099999904632568, 1.993899941444397, 2.1199998855590816]   |
| V278     | float           | True       |              77.91 | [0.0, 1.3085999488830566, 1.993899941444397, 2.1517999172210693, 2.2578001022338867]   |
| V331     | float           | True       |              86.05 | [0.0, 5.0, 6.0, 7.0, 7.5]                                                              |
| V332     | float           | True       |              86.05 | [0.0, 1.4800000190734863, 5.0, 6.0, 7.0]                                               |
| V333     | float           | True       |              86.05 | [0.0, 5.0, 6.0, 7.0, 7.5]                                                              |
| V334     | float           | True       |              86.05 | [0.0, 5.0, 6.0, 7.0, 7.5]                                                              |
| V335     | float           | True       |              86.05 | [0.0, 1.4800000190734863, 5.0, 6.0, 7.0]                                               |
| V336     | float           | True       |              86.05 | [0.0, 5.0, 6.0, 7.0, 7.5]                                                              |
| V337     | float           | True       |              86.05 | [0.0, 5.0, 6.0, 7.0, 8.0]                                                              |
| V338     | float           | True       |              86.05 | [0.0, 5.0, 6.0, 7.0, 7.5]                                                              |
| V339     | float           | True       |              86.05 | [0.0, 5.0, 6.0, 7.0, 8.0]                                                              |
| V159     | float           | True       |              86.12 | [0.0, 5.0, 6.0, 7.0, 8.0]                                                              |
| V160     | float           | True       |              86.12 | [0.0, 5.0, 6.0, 7.0, 7.5]                                                              |
| V161     | float           | True       |              86.12 | [0.0, 5.0, 7.0, 7.5, 8.0]                                                              |
| V162     | float           | True       |              86.12 | [0.0, 5.0, 7.0, 7.5, 8.0]                                                              |
| V163     | float           | True       |              86.12 | [0.0, 5.0, 7.0, 7.5, 8.0]                                                              |
| V164     | float           | True       |              86.12 | [0.0, 5.0, 6.0, 8.0, 10.0]                                                             |
| V165     | float           | True       |              86.12 | [0.0, 5.0, 6.0, 8.0, 10.0]                                                             |
| V166     | float           | True       |              86.12 | [0.0, 5.0, 6.0, 7.0, 8.0]                                                              |
| V279     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V280     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V284     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V285     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V286     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V287     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V290     | integer         | True       |               0    | [1.0, 2.0, 3.0, 4.0, 5.0]                                                              |
| V291     | integer         | True       |               0    | [1.0, 2.0, 3.0, 4.0, 5.0]                                                              |
| V292     | integer         | True       |               0    | [1.0, 2.0, 3.0, 4.0, 5.0]                                                              |
| V293     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V294     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V295     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V297     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V298     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V299     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V302     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V303     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V304     | integer         | True       |               0    | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V305     | integer         | True       |               0    | [1.0, 2.0]                                                                             |
| V100     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V101     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V102     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V103     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V104     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V105     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V106     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V108     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V109     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V110     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V111     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V112     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V113     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V114     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V115     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V116     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V117     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0]                                                                   |
| V118     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0]                                                                   |
| V119     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0]                                                                   |
| V120     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0]                                                                   |
| V121     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0]                                                                   |
| V122     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0]                                                                   |
| V123     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V124     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V125     | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V95      | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V96      | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V97      | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V98      | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V99      | integer         | True       |               0.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V281     | integer         | True       |               0.21 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V282     | integer         | True       |               0.21 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V283     | integer         | True       |               0.21 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V288     | integer         | True       |               0.21 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V289     | integer         | True       |               0.21 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V296     | integer         | True       |               0.21 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V300     | integer         | True       |               0.21 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V301     | integer         | True       |               0.21 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V12      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0]                                                                   |
| V13      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V15      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V16      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V17      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V18      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V19      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V20      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V21      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V22      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V23      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V24      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V25      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V26      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V27      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 4.0]                                                                   |
| V28      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 4.0]                                                                   |
| V29      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V30      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V31      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V32      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V33      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V34      | integer         | True       |              12.88 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V53      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V54      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V55      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V56      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V57      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V58      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V59      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V60      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V61      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V62      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V63      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V64      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V66      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V67      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V68      | integer         | True       |              13.06 | [0.0, 1.0, 2.0]                                                                        |
| V69      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V70      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V71      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V72      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V73      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V74      | integer         | True       |              13.06 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V75      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V76      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V77      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V78      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V79      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V80      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V81      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V82      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V83      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V84      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V85      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V86      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V87      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V89      | integer         | True       |              15.1  | [0.0, 1.0, 2.0]                                                                        |
| V90      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V91      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V92      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V93      | integer         | True       |              15.1  | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V94      | integer         | True       |              15.1  | [0.0, 1.0, 2.0]                                                                        |
| V35      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0]                                                                   |
| V36      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V37      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V38      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V39      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V40      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V42      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V43      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V44      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V45      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V46      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V47      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V48      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V49      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V50      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V51      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V52      | integer         | True       |              28.61 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V10      | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V11      | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V2       | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V3       | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V4       | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V5       | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V6       | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V7       | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V8       | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V9       | integer         | True       |              47.29 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V220     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V221     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V222     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V227     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V234     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V238     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V239     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V245     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V250     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V251     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V255     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V256     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V259     | integer         | True       |              76.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V169     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V170     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V171     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V174     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V175     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V180     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V184     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V185     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V188     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V189     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V194     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V195     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V197     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V198     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V200     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V201     | integer         | True       |              76.32 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V167     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V168     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V172     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V173     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V176     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V177     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V178     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V179     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V181     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V182     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V183     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V186     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V187     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V190     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V191     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V192     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V193     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V196     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V199     | integer         | True       |              76.36 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V217     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V218     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V219     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V223     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V224     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V225     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V226     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V228     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V229     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V230     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V231     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V232     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V233     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V235     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V236     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V237     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V240     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 5.0, 6.0]                                                              |
| V241     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 4.0, 5.0]                                                              |
| V242     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V243     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V244     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V246     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V247     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V248     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V249     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V252     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V253     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V254     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V257     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V258     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V260     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V261     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V262     | integer         | True       |              77.91 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V322     | integer         | True       |              86.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V323     | integer         | True       |              86.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V324     | integer         | True       |              86.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V325     | integer         | True       |              86.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V326     | integer         | True       |              86.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V327     | integer         | True       |              86.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V328     | integer         | True       |              86.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V329     | integer         | True       |              86.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V330     | integer         | True       |              86.05 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V138     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V139     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V140     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V141     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V142     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V143     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V144     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V145     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V146     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V147     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V148     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V149     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V150     | integer         | True       |              86.12 | [1.0, 2.0, 3.0, 4.0, 5.0]                                                              |
| V151     | integer         | True       |              86.12 | [1.0, 2.0, 3.0, 4.0, 5.0]                                                              |
| V152     | integer         | True       |              86.12 | [1.0, 2.0, 3.0, 4.0, 5.0]                                                              |
| V153     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V154     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V155     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V156     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V157     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |
| V158     | integer         | True       |              86.12 | [0.0, 1.0, 2.0, 3.0, 4.0]                                                              |